# Generating MME seasonal data
- Takes seasonal data generated by monthly_to_seasonal_data_generation.py and computes the Multi Model Ensemble Mean for each region


In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import glob
import os

# define folder path to seasonal data (change to your path)
seasonal_data_path = 'data/csv/'

In [ ]:
'''Computing MME for each region'''

# Initialize an empty dictionary to store DataFrames
dfs_dict = {}

# Iterate over every file in the seasonal data folder
for file_name in os.listdir(seasonal_data_path):

    # Skip files with 'MME' or 'SMME' in the name
    if 'MME' in file_name or 'SMME' in file_name:
        print(f"Skipping file with 'MME' or 'SMME': {file_name}")
        continue

    # Build the full file path
    file_path = os.path.join(seasonal_data_path, file_name)

    # Read the CSV file
    df = pd.read_csv(file_path)

    # Store the DataFrame using the file name as key
    dfs_dict[file_name] = df

# concatenate all dataframes, and compute the mean predicted and actual precip for each unique combination of season, region, lead time, etc.
mme_df = pd.concat(dfs_dict.values()).groupby(['season', 'region', 'realization_year', 'year_of_prediction', 'month_of_prediction', 'lead_time'])[['predicted_precip','precip']].mean().reset_index()

# create a model column with value MME
mme_df['model'] = 'MME'

In [ ]:
'''Saving MME data for each region individually'''

# separate the regions, save a file for each region
# i.e lake_victoria_basin_MME_merged_seasonal.csv
for region, region_df in mme_df.groupby('region'):

    # get the region name
    region_name = str(region)

    # build save file path
    save_file_path = os.path.join(seasonal_data_path, f'{region_name}_MME_merged_seasonal.csv')

    # save file
    region_df.to_csv(save_file_path, index=False)